In [33]:
import numpy as np
from PIL import Image
import heapq
from collections import defaultdict
from typing import List, Tuple, Set, Dict, Sequence

def load_image(image_path: str) -> np.ndarray:
    """Загрузка изображения и преобразование в numpy-массив"""
    img = Image.open(image_path)
    return np.array(img)

def build_graph(pixels: np.ndarray) -> List[Tuple[float, int, int]]:
    """Построение графа (список рёбер) на основе разницы цветов пикселей"""
    h, w, _ = pixels.shape
    edges: List[Tuple[float, int, int]] = []
    
    # Проходим по всем пикселям и добавляем рёбра между соседями (4-связность)
    for i in range(h):
        for j in range(w):
            current_pos = i * w + j 
            current_color = pixels[i, j].astype(float)
            
            # Проверяем соседей (вправо и вниз, чтобы избежать дублирования)
            for di, dj in [(0, 1), (1, 0)]:
                ni, nj = i + di, j + dj
                if ni < h and nj < w:
                    neighbor_pos = ni * w + nj
                    neighbor_color = pixels[ni, nj].astype(float)
                    
                    # Нормализованное евклидово расстояние между цветами (0-1)
                    distance = np.sqrt(np.sum((current_color - neighbor_color) ** 2)) / (255 * np.sqrt(3))
                    edges.append((distance, current_pos, neighbor_pos))
    
    return edges

def prim_mst(edges: List[Tuple[float, int, int]], num_pixels: int) -> List[Tuple[int, int, float]]:
    """Алгоритм Прима для построения минимального остовного дерева"""
    # Инициализируем структуры данных
    adjacency: List[List[Tuple[int, float]]] = [[] for _ in range(num_pixels)]
    for d, u, v in edges:
        adjacency[u].append((v, d))
        adjacency[v].append((u, d))
    
    visited: List[bool] = [False] * num_pixels
    mst_edges: List[Tuple[int, int, float]] = []
    heap: List[Tuple[float, int, int]] = []
    
    # Начинаем с вершины 0
    visited[0] = True
    for v, d in adjacency[0]:
        heapq.heappush(heap, (d, 0, v))
    
    while heap and len(mst_edges) < num_pixels - 1:
        d, u, v = heapq.heappop(heap)
        if not visited[v]:
            visited[v] = True
            mst_edges.append((u, v, d))
            for neighbor, neighbor_d in adjacency[v]:
                if not visited[neighbor]:
                    heapq.heappush(heap, (neighbor_d, v, neighbor))
    
    return mst_edges

def layer_clustering_mst(mst_edges: List[Tuple[int, int, float]], 
                        thresholds: Sequence[float]) -> List[List[Set[int]]]:
    """
    Послойная кластеризация на основе MST:
    Для каждого порога удаляем все рёбра, превышающие этот порог
    """
    # Сортируем пороги по возрастанию
    sorted_thresholds = sorted(thresholds)
    clusters_history: List[List[Set[int]]] = []
    
    for threshold in sorted_thresholds:
        # Строим граф без рёбер, превышающих порог
        graph: Dict[int, List[int]] = defaultdict(list)
        for u, v, d in mst_edges:
            if d <= threshold:
                graph[u].append(v)
                graph[v].append(u)
        
        # Находим связные компоненты (кластеры)
        visited: Set[int] = set()
        clusters: List[Set[int]] = []
        num_pixels = max(max(u, v) for u, v, _ in mst_edges) + 1
        
        for node in range(num_pixels):
            if node not in visited:
                stack = [node]
                cluster: Set[int] = set()
                while stack:
                    current = stack.pop()
                    if current not in visited:
                        visited.add(current)
                        cluster.add(current)
                        for neighbor in graph[current]:
                            if neighbor not in visited:
                                stack.append(neighbor)
                clusters.append(cluster)
        
        clusters_history.append(clusters)
    
    return clusters_history

# def visualize_clusters(pixels: np.ndarray, 
#                      clusters: List[List[Set[int]]], 
#                      threshold_index: int) -> np.ndarray:
#     """Визуализация кластеров: каждому кластеру присваивается случайный цвет."""
#     h, w, _ = pixels.shape
#     output: np.ndarray = np.zeros_like(pixels)
    
#     clusters_at_level = clusters[threshold_index]
#     for cluster in clusters_at_level:
#         # Генерируем случайный цвет для кластера
#         color = np.random.randint(0, 256, size=3)
        
#         for pos in cluster:
#             i, j = pos // w, pos % w
#             output[i, j] = color
    
#     return output

In [34]:
def visualize_clusters(pixels: np.ndarray, 
                     clusters: List[List[Set[int]]], 
                     threshold_index: int) -> np.ndarray:
    """Визуализация кластеров: каждому кластеру присваивается средний цвет."""
    h, w, _ = pixels.shape
    output: np.ndarray = np.zeros_like(pixels)
    
    clusters_at_level: List[Set[int]] = clusters[threshold_index]
    
    for cluster in clusters_at_level:
        # Собираем все пиксели кластера
        cluster_pixels: List[np.ndarray] = []
        positions: List[Tuple[int, int]] = []
        
        for pos in cluster:
            i, j = pos // w, pos % w
            cluster_pixels.append(pixels[i, j])
            positions.append((i, j))
        
        # Вычисляем средний цвет кластера
        if cluster_pixels: 
            mean_color: np.ndarray = np.mean(cluster_pixels, axis=0).astype(np.uint8)
            
            # Закрашиваем все пиксели кластера средним цветом
            for i, j in positions:
                output[i, j] = mean_color
    
    return output

In [36]:
# Параметры
input_path = "origins/sk.jpg"
output_path = "results/task_2/sk_clustered.jpg"
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]

# Загрузка изображения
pixels = load_image(input_path)
h, w, _ = pixels.shape
num_pixels = h * w

# Построение графа
edges = build_graph(pixels)

# Построение MST
mst_edges = prim_mst(edges, num_pixels)

# Выполняем послойную кластеризацию
clusters = layer_clustering_mst(mst_edges, thresholds)

# Визуализация
clustered_image = visualize_clusters(pixels, clusters, 0)

# Сохраняем результат
result_img = Image.fromarray(clustered_image)
result_img.save(output_path)
result_img.show()